# Semantic segmantation

- U-Net을 통한 세그멘테이션 작업이 정상적으로 진행되었는가?  
&rarr; KITTI 데이터셋 구성, U-Net 모델 훈련, 결과물 시각화의 한 사이클이 정상 수행되어 세그멘테이션 결과 이미지를 제출하였다.  
- U-Net++ 모델이 성공적으로 구현되었는가?  
&rarr; U-Net++ 모델을 스스로 구현하여 학습 진행 후 세그멘테이션 결과까지 정상 진행되었다.  
- U-Net과 U-Net++ 두 모델의 성능이 정량적/정성적으로 잘 비교되었는가?  
&rarr; U-Net++ 의 세그멘테이션 결과 사진과 IoU 계산치를 U-Net과 비교하여 우월함을 확인하였다.  

# 0. 라이브러리 import & setting

In [4]:
import os
from glob import glob # 파일, 디렉터리 찾을 때 사용
import numpy as np

from PIL import Image
import matplotlib.pyplot as plt
from skimage.io import imread # 이미지 파일 불러서 numpy array로 변환 (H, W, C), C: RGB
from skimage.transform import resize

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

from albumentations import HorizontalFlip, RandomSizedCrop, Compose, OneOf, Resize

In [6]:
DATA_DIR = os.path.join(os.getcwd(), "data/training")
print(DATA_DIR)

/home/minkyujeong/work/DeepDive/ch3_semantic_seg/data/training


In [27]:
BATCH_SIZE = 16

# 1. 데이터셋 준비 (KITTI 데이터셋 구성)

In [ ]:
# data augmentation 수행 함수
def augment_data(is_train=True): # 학습 데이터만 True
    if is_train: # 학습 데이터일 때 augmentation
        return Compose([
            HorizontalFlip(p=0.5), # 0.5의 확률로 h-flip
            RandomSizedCrop(                
                min_max_height=(300,370), # 높이를 300~370 중 선택
                w2h_ratio=1242/370, # h에 대한 w 비율
                # h:2, w:1 이면 w2h는 0.5, h:4가 되면 w = 4 * 0.5 = 2
                # 원본 이미지에서 h:370, w:1242 -> new_w = new_h * w2h
                size=(224, 224), # h, w
                p=0.5
            ),
            # RandomSizedCrop의 resize는 0.5 확률로 작동해서
            Resize(width=224, height=224)
        ])
    else: # 테스트 데이터는 resize만
        return Compose([Resize(width=224, height=224)])

In [25]:
class KittiDataset(Dataset):
    def __init__(self, data_path, img_size=(224,224,3), out_size=(224,224), is_train=True, aug=None):
        '''
        data_path: data 디렉터리 path
        img_size: preprocess에 사용할 입력 이미지의 크기
        out_size: ground truth를 만들어주기 위한 크기
        is_trian: 학습용인지 구분
        aug: 적용하려는 augmentation 함수
        '''
        super().__init__()
        self.data_path = data_path
        self.img_size = img_size
        self.out_size = out_size,
        self.is_train = is_train
        self.aug = aug

        # load_data를 통해 KITTI dataset의 경로에서 이미지, 라벨 확인
        self.data = self.load_data()

    def load_data(self):
        # KITTI dataset에서 필요한 정보(이미지 경로, 라벨 경로)를 실제 디렉터리에서 확인하고 load
        img_paths = sorted(glob(os.path.join(self.data_path, "image_2", "*.png")))
        label_paths = sorted(glob(os.path.join(self.data_path, "semantic", "*.png")))

        assert len(img_paths) == len(label_paths), print("이미지, 라벨 개수를 확인하세요..")
        data = list(zip(img_paths, label_paths))

        if self.is_train: # train data 일 때 
            return data[:-30] # 전체 데이터에서 30개 빼고 train data
        return data[-30:] # 30개는 validation data
    
    def __len__(self): # dataset 개수
        return len(self.data)
    
    def __getitem__(self,idx):
        # image와 label을 반환
        # image는 resize 및 augmentation 적용된 image
        # label은 resize만 적용한 semantic image
        img_path, label_path = self.data[idx]

        _img = imread(img_path) # numpy.ndarray (H, W, C), 0~255 값 
        _label = imread(label_path) # numpy.ndarray (H, W), 0~255값 
        
        # 도로에 해당하는 라벨을 마스크로 변환 (uint8: 0~255)
        _label = (_label == 7).astype(np.uint8) * 1 # 도로에 해당하는 픽셀은 1, 아니면 0인 이미지가 됨
        
        data = {'image': _img, 'mask': _label}
        
        # aug 적용
        if self.aug: # train은 여러 augmentation, test는 resize만
            augmented = self.aug(**data)
            _img = augmented['image'] / 255.0 # normalize
            _label = augmented['mask'] # 1 or 0
        
        # 마스크는 1채널이라 2차원 데이터로 존재함 (image는 3차원)
        # 마스크 차원 확장 (H, W) -> (C, H, W)
        _label = np.expand_dims(_label, axis=0)

        # torch.tensor로 변환한 후 tuple로 반환
        return (torch.tensor(_img, dtype=torch.float32).permute(2,0,1), # (H, W, C) -> (C, H, W)
                torch.tensor(_label, dtype=torch.float32)) # (1, H, W)
    
    def shuffle_data(self):
        # 한 에폭이 끝나면 실행 (학습 중인 경우, 데이터를 random shuffle)
        if self.is_train:
            np.random.shuffle(self.data)

In [20]:
sample_img_path = glob(os.path.join(DATA_DIR, "image_2", "*.png"))
sample_path = glob(os.path.join(DATA_DIR, "semantic", "*.png"))
sample_path[0]

'/home/minkyujeong/work/DeepDive/ch3_semantic_seg/data/training/semantic/000042_10.png'

In [24]:
a_img = imread(sample_img_path[0])
print(a_img.dtype)
print(a_img)

uint8
[[[111 175 254]
  [111 176 253]
  [111 177 254]
  ...
  [ 97 157 228]
  [ 98 157 234]
  [100 158 233]]

 [[111 179 255]
  [111 181 254]
  [112 180 255]
  ...
  [ 98 157 229]
  [ 98 157 235]
  [ 99 157 234]]

 [[112 179 255]
  [112 180 254]
  [113 179 255]
  ...
  [ 98 157 233]
  [ 99 157 239]
  [100 156 237]]

 ...

 [[ 91  75  74]
  [ 84  85 100]
  [101 100 113]
  ...
  [ 64  71  47]
  [ 62  69  48]
  [ 60  70  47]]

 [[ 82  62  65]
  [ 79  79  83]
  [105  93  85]
  ...
  [ 64  66  38]
  [ 62  65  37]
  [ 60  65  39]]

 [[101  76  61]
  [100  85  82]
  [113  96  86]
  ...
  [ 63  65  39]
  [ 59  64  36]
  [ 55  63  37]]]


In [19]:
# type(imread(sample_path[0]))
a = imread(sample_path[0])
print(a)
(a == 7).astype(np.uint8)


[[23 23 23 ... 23 23 23]
 [23 23 23 ... 23 23 23]
 [23 23 23 ... 23 23 23]
 ...
 [ 7  7  7 ... 22 22 22]
 [ 7  7  7 ... 22 22 22]
 [ 7  7  7 ... 22 22 22]]


array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0]], shape=(375, 1242), dtype=uint8)

In [26]:
# train, test 각각에 대한 preprocess 할당
train_aug = augment_data()
test_aug = augment_data(is_train=False)

train_dataset = KittiDataset(DATA_DIR, is_train=True, aug=train_aug)
test_dataset = KittiDataset(DATA_DIR, is_train=False, aug=test_aug)

In [28]:
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 2. 모델 정의

## 2.1. UNet

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_chs=3, out_chs=1):
        super().__init__()
        


## 2.2. UNet++

# 3. 모델 학습

## 3.1. UNet

## 3.2. Unet++

# 4. 결과 시각화

## 4.1. UNet

## 4.2. UNet++

# 5. 성능 비교

## 5.1. 정량적 비교

## 5.2. 정성적 비교

# 회고